# ECAPA-TDNN Speaker Verification - Baseline Evaluation

**Model**: ECAPA-TDNN (SpeechBrain, pre-trained on VoxCeleb)

**Dataset**: 25 speakers x 7 prompts = 175 audio clips

## 1. Install Dependencies

In [ ]:
!pip install -q speechbrain torch torchaudio soundfile librosa matplotlib scikit-learn seaborn

## 2. Imports

In [ ]:
import os
import numpy as np
import torch
import torchaudio
import soundfile as sf
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics import roc_curve, auc
from numpy.linalg import norm
import warnings
warnings.filterwarnings('ignore')

if not hasattr(torchaudio, 'list_audio_backends'):
    torchaudio.list_audio_backends = lambda: ['soundfile']

from speechbrain.inference.speaker import EncoderClassifier
print('All imports OK')

## 3. Upload Dataset
Upload your `voice-set.zip` when prompted.

In [ ]:
import zipfile
from google.colab import files

print('Upload voice-set.zip...')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')

EXTRACT_DIR = 'voice-set'
speakers = sorted([d for d in os.listdir(EXTRACT_DIR)
                    if os.path.isdir(os.path.join(EXTRACT_DIR, d))])
print('Found', len(speakers), 'speakers:')
for i, s in enumerate(speakers):
    n = len([f for f in os.listdir(os.path.join(EXTRACT_DIR, s)) if f.endswith('.ogg')])
    print('  [%02d] %s (%d clips)' % (i, s, n))

## 4. Load ECAPA-TDNN Model

In [ ]:
print('Loading ECAPA-TDNN...')

import huggingface_hub
if not hasattr(huggingface_hub, '_original_download'):
    huggingface_hub._original_download = huggingface_hub.hf_hub_download
    def patched_download(*args, **kwargs):
        kwargs.pop('use_auth_token', None)
        return huggingface_hub._original_download(*args, **kwargs)
    huggingface_hub.hf_hub_download = patched_download

from speechbrain.inference.speaker import EncoderClassifier
model = EncoderClassifier.from_hparams(
    source='speechbrain/spkrec-ecapa-voxceleb',
    savedir='pretrained_models/spkrec-ecapa-voxceleb',
)
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
model.to(device)
total_params = sum(p.numel() for p in model.mods.embedding_model.parameters())
print('Loaded! Params:', format(total_params, ','), '| Embedding dim: 192')


## 5. Extract Embeddings

In [ ]:
def load_audio(path, sr=16000):
    data, orig_sr = sf.read(path)
    if len(data.shape) > 1:
        data = np.mean(data, axis=1)
    if orig_sr != sr:
        import librosa
        data = librosa.resample(data, orig_sr=orig_sr, target_sr=sr)
    return data, sr

def get_emb(model, audio):
    sig = torch.FloatTensor(audio).unsqueeze(0)
    with torch.no_grad():
        e = model.encode_batch(sig)
    return e.squeeze().numpy()

all_embeddings = []
all_labels = []
all_speaker_names = []
all_clip_nums = []
all_audio_data = {}
speaker_embeddings = defaultdict(list)

for i, speaker in enumerate(speakers):
    sp_dir = os.path.join(EXTRACT_DIR, speaker)
    oggs = sorted([f for f in os.listdir(sp_dir) if f.endswith('.ogg')])
    for ogg in oggs:
        cn = ogg.replace('.ogg', '')
        clip_num = int(cn) if cn.isdigit() else 0
        fpath = os.path.join(sp_dir, ogg)
        ad, sr = load_audio(fpath)
        emb = get_emb(model, ad)
        all_embeddings.append(emb)
        all_labels.append(i)
        all_speaker_names.append(speaker)
        all_clip_nums.append(clip_num)
        all_audio_data[(speaker, clip_num)] = (ad, sr)
        speaker_embeddings[speaker].append(emb)
    print('  [%02d] %s: %d clips' % (i, speaker, len(oggs)))

all_embeddings = np.array(all_embeddings)
all_labels = np.array(all_labels)
print('Total embeddings:', all_embeddings.shape)

## 6. Waveforms & Spectrograms

In [ ]:
fig, axes = plt.subplots(5, 4, figsize=(20, 15))
fig.suptitle('Waveforms & Mel-Spectrograms', fontsize=18, fontweight='bold')

for row, speaker in enumerate(speakers[:5]):
    ad1, sr1 = all_audio_data[(speaker, 1)]
    ad4, sr4 = all_audio_data.get((speaker, 4), (ad1, sr1))

    axes[row, 0].plot(np.arange(len(ad1))/sr1, ad1, color='#2196F3', linewidth=0.5)
    axes[row, 0].set_title(speaker + ' - Clip 1', fontsize=10)
    axes[row, 0].set_xlabel('Time (s)')
    axes[row, 0].set_ylabel('Amp')
    axes[row, 0].set_ylim(-1, 1)

    mel1 = torchaudio.transforms.MelSpectrogram(sample_rate=sr1, n_mels=80, n_fft=400, hop_length=160)(
        torch.FloatTensor(ad1).unsqueeze(0))
    axes[row, 1].imshow(torchaudio.transforms.AmplitudeToDB()(mel1).squeeze().numpy(),
                         aspect='auto', origin='lower', cmap='magma')
    axes[row, 1].set_title(speaker + ' - Mel-Spec', fontsize=10)

    axes[row, 2].plot(np.arange(len(ad4))/sr4, ad4, color='#FF5722', linewidth=0.5)
    axes[row, 2].set_title(speaker + ' - Clip 4', fontsize=10)
    axes[row, 2].set_xlabel('Time (s)')
    axes[row, 2].set_ylim(-1, 1)

    mel4 = torchaudio.transforms.MelSpectrogram(sample_rate=sr4, n_mels=80, n_fft=400, hop_length=160)(
        torch.FloatTensor(ad4).unsqueeze(0))
    axes[row, 3].imshow(torchaudio.transforms.AmplitudeToDB()(mel4).squeeze().numpy(),
                         aspect='auto', origin='lower', cmap='magma')
    axes[row, 3].set_title(speaker + ' - Clip 4 Mel', fontsize=10)

plt.tight_layout()
plt.savefig('01_waveforms.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. t-SNE Clustering
Each dot = one clip. Same color = same speaker.

In [ ]:
perp = min(30, len(all_embeddings) - 1)
tsne = TSNE(n_components=2, random_state=42, perplexity=perp, n_iter=1000, learning_rate='auto')
emb_2d = tsne.fit_transform(all_embeddings)

cmap_c = plt.cm.tab20(np.linspace(0, 1, 20))
extra_c = plt.cm.Set3(np.linspace(0, 1, 12))
palette = np.vstack([cmap_c, extra_c[:5]])

fig, ax = plt.subplots(figsize=(16, 12))
fig.suptitle('t-SNE of Speaker Embeddings (Baseline)', fontsize=18, fontweight='bold')
for i, sp in enumerate(speakers):
    mask = all_labels == i
    ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1], c=[palette[i]], label=sp,
               s=100, alpha=0.8, edgecolors='white', linewidths=0.5)
    cx = np.mean(emb_2d[mask, 0])
    cy = np.mean(emb_2d[mask, 1])
    ax.annotate(sp, (cx, cy), fontsize=7, fontweight='bold', ha='center',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))
ax.set_xlabel('t-SNE Dim 1'); ax.set_ylabel('t-SNE Dim 2')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('02_tsne.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Speaker Similarity Heatmap
25x25 matrix showing cosine similarity between speaker averages.

In [ ]:
avg_embs = np.array([np.mean(speaker_embeddings[s], axis=0) for s in speakers])
normed = avg_embs / (norm(avg_embs, axis=1, keepdims=True) + 1e-9)
sim_mat = normed @ normed.T

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(sim_mat, annot=True, fmt='.2f', cmap='RdYlGn',
            xticklabels=speakers, yticklabels=speakers,
            vmin=-0.1, vmax=1.0, center=0.3, square=True,
            linewidths=0.5, annot_kws={'size': 6}, ax=ax)
ax.set_title('Speaker Similarity Heatmap', fontsize=14, fontweight='bold')
ax.set_xticklabels(speakers, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(speakers, rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('03_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Same vs Different Speaker Scores
Green = same person. Red = different people. Less overlap = better.

In [ ]:
cosine = torch.nn.CosineSimilarity(dim=0)
same_scores = []
diff_scores = []
for i in range(len(all_embeddings)):
    for j in range(i + 1, len(all_embeddings)):
        s = cosine(torch.tensor(all_embeddings[i]), torch.tensor(all_embeddings[j])).item()
        if all_labels[i] == all_labels[j]:
            same_scores.append(s)
        else:
            diff_scores.append(s)

print('Same pairs:', len(same_scores))
print('Diff pairs:', len(diff_scores))
print('Avg same: %.4f' % np.mean(same_scores))
print('Avg diff: %.4f' % np.mean(diff_scores))
print('Gap:      %.4f' % (np.mean(same_scores) - np.mean(diff_scores)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
ax1.hist(same_scores, bins=40, alpha=0.7, color='#4CAF50', label='Same Speaker', density=True)
ax1.hist(diff_scores, bins=40, alpha=0.7, color='#F44336', label='Diff Speaker', density=True)
ax1.axvline(x=0.50, color='#FF9800', linestyle='--', linewidth=2, label='Threshold')
ax1.set_xlabel('Cosine Similarity'); ax1.set_ylabel('Density')
ax1.set_title('Score Distribution', fontsize=14, fontweight='bold')
ax1.legend(); ax1.grid(True, alpha=0.3)

bp = ax2.boxplot([same_scores, diff_scores], labels=['Same', 'Different'],
                  patch_artist=True, showmeans=True,
                  meanprops=dict(marker='D', markerfacecolor='gold', markersize=8))
bp['boxes'][0].set_facecolor('#4CAF50'); bp['boxes'][0].set_alpha(0.7)
bp['boxes'][1].set_facecolor('#F44336'); bp['boxes'][1].set_alpha(0.7)
ax2.axhline(y=0.50, color='#FF9800', linestyle='--', linewidth=2, label='Threshold')
ax2.set_ylabel('Cosine Similarity')
ax2.set_title('Box Plot', fontsize=14, fontweight='bold')
ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('04_scores.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Full 175x175 Clip Matrix

In [ ]:
n = len(all_embeddings)
full_sim = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        full_sim[i, j] = cosine(torch.tensor(all_embeddings[i]), torch.tensor(all_embeddings[j])).item()

fig, ax = plt.subplots(figsize=(18, 16))
im = ax.imshow(full_sim, cmap='RdYlGn', vmin=-0.1, vmax=1.0, aspect='auto')
boundaries = []
cur = all_labels[0]
for i in range(1, len(all_labels)):
    if all_labels[i] != cur:
        boundaries.append(i - 0.5)
        cur = all_labels[i]
for b in boundaries:
    ax.axhline(y=b, color='black', linewidth=0.5, alpha=0.5)
    ax.axvline(x=b, color='black', linewidth=0.5, alpha=0.5)

lp = []; st = 0
for i in range(1, len(all_labels)):
    if all_labels[i] != all_labels[i-1]:
        lp.append(((st + i - 1) / 2, speakers[all_labels[st]])); st = i
lp.append(((st + len(all_labels) - 1) / 2, speakers[all_labels[st]]))
ax.set_xticks([p for p, _ in lp])
ax.set_xticklabels([nm for _, nm in lp], rotation=45, ha='right', fontsize=7)
ax.set_yticks([p for p, _ in lp])
ax.set_yticklabels([nm for _, nm in lp], fontsize=7)
ax.set_title('Full Clip-to-Clip Similarity', fontsize=16, fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.savefig('05_full_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Per-Speaker Consistency
Bar chart showing voice consistency per speaker.

In [ ]:
intra = {}
for sp in speakers:
    embs = speaker_embeddings[sp]
    sc = []
    for i in range(len(embs)):
        for j in range(i+1, len(embs)):
            sc.append(cosine(torch.tensor(embs[i]), torch.tensor(embs[j])).item())
    intra[sp] = np.mean(sc) if sc else 0

fig, ax = plt.subplots(figsize=(16, 7))
vals = list(intra.values())
cols = ['#4CAF50' if v > 0.5 else '#FF9800' if v > 0.3 else '#F44336' for v in vals]
bars = ax.bar(range(len(speakers)), vals, color=cols, alpha=0.8, edgecolor='white')
ax.axhline(y=0.50, color='red', linestyle='--', linewidth=1.5, label='Threshold')
ax.axhline(y=np.mean(vals), color='blue', linestyle=':', linewidth=1.5,
           label='Mean (%.3f)' % np.mean(vals))
ax.set_xticks(range(len(speakers)))
ax.set_xticklabels(speakers, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Avg Intra-Speaker Similarity')
ax.set_title('Per-Speaker Voice Consistency', fontsize=14, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2., b.get_height() + 0.01,
            '%.2f' % v, ha='center', va='bottom', fontsize=7, fontweight='bold')
plt.tight_layout()
plt.savefig('06_consistency.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. ROC & DET Curves

In [ ]:
y_true = []
y_pred = []
for i in range(len(all_embeddings)):
    for j in range(i + 1, len(all_embeddings)):
        s = cosine(torch.tensor(all_embeddings[i]), torch.tensor(all_embeddings[j])).item()
        y_pred.append(s)
        y_true.append(1 if all_labels[i] == all_labels[j] else 0)
y_true = np.array(y_true)
y_pred = np.array(y_pred)

fpr, tpr, thresholds = roc_curve(y_true, y_pred)
roc_auc = auc(fpr, tpr)
fnr = 1 - tpr
eer_idx = np.nanargmin(np.abs(fpr - fnr))
eer = (fpr[eer_idx] + fnr[eer_idx]) / 2
eer_thr = thresholds[eer_idx]
print('AUC: %.4f | EER: %.2f%% at threshold %.3f' % (roc_auc, eer*100, eer_thr))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
ax1.plot(fpr, tpr, color='#2196F3', linewidth=2, label='ROC (AUC=%.4f)' % roc_auc)
ax1.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5)
ax1.scatter([fpr[eer_idx]], [tpr[eer_idx]], color='red', s=100, zorder=5, label='EER=%.4f' % eer)
ax1.set_xlabel('FPR'); ax1.set_ylabel('TPR')
ax1.set_title('ROC Curve', fontsize=14, fontweight='bold')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(fpr * 100, fnr * 100, color='#FF5722', linewidth=2, label='DET')
ax2.plot([0, 100], [0, 100], 'k--', linewidth=1, alpha=0.5)
ax2.scatter([fpr[eer_idx]*100], [fnr[eer_idx]*100], color='red', s=100, zorder=5,
            label='EER=%.2f%%' % (eer*100))
ax2.set_xlabel('FPR (%)'); ax2.set_ylabel('FNR (%)')
ax2.set_title('DET Curve', fontsize=14, fontweight='bold')
ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('07_roc_det.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Embedding Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Embedding Analysis', fontsize=18, fontweight='bold')

ax = axes[0, 0]
disp = np.array([np.mean(speaker_embeddings[s], axis=0) for s in speakers[:10]])
im1 = ax.imshow(disp, aspect='auto', cmap='coolwarm')
ax.set_yticks(range(10)); ax.set_yticklabels(speakers[:10], fontsize=9)
ax.set_xlabel('Dimensions (0-191)'); ax.set_title('Avg Embedding (10 speakers)')
plt.colorbar(im1, ax=ax)

ax = axes[0, 1]
ax.bar(range(192), np.var(all_embeddings, axis=0), color='#2196F3', alpha=0.7)
ax.set_xlabel('Dimension'); ax.set_ylabel('Variance')
ax.set_title('Per-Dimension Variance')

ax = axes[1, 0]
norms = np.linalg.norm(all_embeddings, axis=1)
ax.hist(norms, bins=30, color='#9C27B0', alpha=0.7, edgecolor='white')
ax.set_xlabel('L2 Norm'); ax.set_ylabel('Count')
ax.set_title('Embedding Norms (mean=%.2f)' % np.mean(norms))

ax = axes[1, 1]
pca = PCA(n_components=50).fit(all_embeddings)
cumvar = np.cumsum(pca.explained_variance_ratio_) * 100
ax.plot(range(1, 51), cumvar, 'o-', color='#FF5722', markersize=3)
n95 = int(np.argmax(cumvar >= 95)) + 1
ax.axhline(y=95, color='gray', linestyle='--', alpha=0.5, label='95%%')
ax.axvline(x=n95, color='green', linestyle='--', alpha=0.5, label='%d dims' % n95)
ax.set_xlabel('Components'); ax.set_ylabel('Variance (%%)')
ax.set_title('PCA Dimensionality'); ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('08_embedding.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. Summary

In [ ]:
print('=' * 55)
print('  BASELINE EVALUATION SUMMARY')
print('=' * 55)
print('  Model:      ECAPA-TDNN (SpeechBrain)')
print('  Trained on: VoxCeleb (7363 speakers)')
print('  Params:    ', format(total_params, ','))
print('  Embedding:  192 dimensions')
print('  Speakers:  ', len(speakers))
print('  Clips:     ', len(all_embeddings))
print('  Avg Same:   %.4f' % np.mean(same_scores))
print('  Avg Diff:   %.4f' % np.mean(diff_scores))
print('  Gap:        %.4f' % (np.mean(same_scores) - np.mean(diff_scores)))
print('  EER:        %.2f%%' % (eer * 100))
print('  ROC AUC:    %.4f' % roc_auc)
print('=' * 55)